# Setup 

In [3]:
from pathlib import Path
import os

SRCDIR = "tablediffusion"
DIR = Path("stuff")
DATADIR = Path("/home/ericwang/TableDiffusion2/data")
RESULTDIR = DIR / "results"

!pwd

for p in [SRCDIR, DIR, DATADIR, RESULTDIR]:
    if not os.path.exists(p):
        print(f"{p} does not exist")

/home/ericwang/TableDiffusion2


In [2]:
!python3 --version

Python 3.12.8


In [2]:
!nvidia-smi

Sat Jan 11 16:59:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000001:00:00.0 Off |                    0 |
| N/A   32C    P0             52W /  300W |       1MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# %%capture
from __future__ import print_function
import argparse
import os
from datetime import datetime
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim as optim
import torch.utils.data
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import HTML
import seaborn as sns
from tqdm import tqdm
import mlflow

In [5]:
# Decide which device we want to run on
DEVICE = torch.device("cuda:0" if (torch.cuda.is_available()) else "cpu")
print(DEVICE)

cuda:0


In [6]:
# Set random seed for reproducibility
SEED = 999
# SEED = random.randint(1, 10000) # use if you want new results
print("Random (meta) seed: ", SEED)

Random (meta) seed:  999


# Synthesis loop

Data synthesis over multiple synthesisers and datasets, with persistence.

In [7]:
%load_ext autoreload
%autoreload 2

In [8]:
import sys
import importlib

sys.path.append(str(SRCDIR))
print(sys.path)

['/home/ericwang/miniconda3/envs/table_diff/lib/python312.zip', '/home/ericwang/miniconda3/envs/table_diff/lib/python3.12', '/home/ericwang/miniconda3/envs/table_diff/lib/python3.12/lib-dynload', '', '/home/ericwang/miniconda3/envs/table_diff/lib/python3.12/site-packages', 'tablediffusion']


In [10]:
from tablediffusion.models import *
import tablediffusion.utilities.utils as utils
import tablediffusion.utilities.data_utils as data_utils
import tablediffusion.config.configs as configs
import tablediffusion.config as config

importlib.reload(utils)
importlib.reload(data_utils)
importlib.reload(config)
importlib.reload(configs)

# NOTES: Running with 10.0 budget very overkill. First few batches immediately burn through ~6.0. The rest of the epoch
# slowly accumulates to ~7.9. Then another half epoch was spend going from 7.9 to 8.2, so I'm guessing the benefits of another
# epoch are small. Total time elapsed 3 hr 20 min and training wasn't done. Gonna try doubling batch size (512 -> 1024) and only 1 epoch.

EPOCHS = 2
DIFFUSION_STEPS = 3

synthesisers = {
    "DPWGAN_Synthesiser": (
        WGAN_Synthesiser,
        {
            "batch_size": 1024,
            'gen_lr': 0.005,
            'dis_lr': 0.001,
            "latent_dim": 128,
            'n_critic': 2,
            "epoch_target": EPOCHS,
            "mlflow_logging": False,
            'gen_dims': (512, 512),
            'dis_dims': (512, 512)
        },
        {
            "n_epochs": EPOCHS,
        },
        {
            "use_raw_data": False,
        },
    ),
    "TableDiffusion_Synthesiser": (
        TableDiffusion_Synthesiser,
        {
            "batch_size": 1024,
            "lr": 0.005,
            "dims": (512, 512),
            "mlflow_logging": False,
            "epoch_target": EPOCHS * DIFFUSION_STEPS,
            "diffusion_steps": DIFFUSION_STEPS,
            "predict_noise": True,
        },
        {
            "n_epochs": EPOCHS,
            "verbose": True,
        },
        {
            "use_raw_data": True,
        },
    )
}
synthesisers

[autoreload of tablediffusion.models.dp_attention_gan failed: Traceback (most recent call last):
  File "/home/ericwang/miniconda3/envs/table_diff/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/home/ericwang/miniconda3/envs/table_diff/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/home/ericwang/miniconda3/envs/table_diff/lib/python3.12/importlib/__init__.py", line 121, in reload
    raise ImportError(f"parent {parent_name!r} not in sys.modules",
ImportError: parent 'tablediffusion.models' not in sys.modules
]


{'DPWGAN_Synthesiser': (models.dp_wgan.WGAN_Synthesiser,
  {'batch_size': 1024,
   'gen_lr': 0.005,
   'dis_lr': 0.001,
   'latent_dim': 128,
   'n_critic': 2,
   'epoch_target': 1,
   'mlflow_logging': False,
   'gen_dims': (512, 512),
   'dis_dims': (512, 512)},
  {'n_epochs': 1},
  {'use_raw_data': False}),
 'TableDiffusion_Synthesiser': (models.table_diffusion.TableDiffusion_Synthesiser,
  {'batch_size': 1024,
   'lr': 0.005,
   'dims': (512, 512),
   'mlflow_logging': False,
   'epoch_target': 3,
   'diffusion_steps': 3,
   'predict_noise': True},
  {'n_epochs': 1, 'verbose': True},
  {'use_raw_data': True})}

In [ ]:
from tablediffusion.utilities import run_synthesisers
import time

dset_name = 'appraise'
datasets = {dset_name: config.datasets[dset_name]}

exp_hash = datetime.now().strftime("%y%m%d_%H%M%S")
EXP_NAME = f"exp_{exp_hash}"

# Make directories for experiment EXP_NAME
EXP_PATH = RESULTDIR / EXP_NAME
# FAKE_DSET_PATH = EXP_PATH / "fake_datasets"
FAKE_DSET_PATH = Path("/home/ericwang/TableDiffusion2/syn/appraise")
if not os.path.exists(FAKE_DSET_PATH):
    os.makedirs(FAKE_DSET_PATH)

exp_id = mlflow.create_experiment(f"{EXP_NAME}")

print(f"\n\nRunning experiment: {EXP_NAME}\n\n")

start = time.time()

run_synthesisers(
    datasets=datasets,
    synthesisers=synthesisers,
    exp_name=EXP_NAME,
    exp_id=exp_id,
    datadir=DATADIR,
    repodir="./",
    epsilon_values=[2.0],
    repeats=1,
    metaseed=SEED,
    generate_fakes=True,
    fake_sample_path=EXP_PATH / "samples",
    fake_data_path=FAKE_DSET_PATH,
    cuda=True,
)

mlflow.end_run()

end = time.time()

print(f'Time Elapsed: {(end - start) / 60} min')



Running experiment: exp_250310_190024


CUDA status: True


Repeats:   0%|                                                                                                                                                                   | 0/1 [00:00<?, ?it/s]

Loaded appraise dataset (15116160, 9) from /home/ericwang/vae_cloud_computing/data/raw/appraise_labeled.csv



ilon:   0%|                                                                                                                                                                   | 0/1 [00:00<?, ?it/s]

:   0%|                                                                                                                                                                    | 0/2 [00:00<?, ?it/s]

Training DPWGAN_Synthesiser on appraise with epsilon=2.0...


...

Sampling t=0, np.sqrt(beta_t)=0.31622776601683794
Sampling t=2, np.sqrt(beta_t)=0.9219544457292886
Sampling t=1, np.sqrt(beta_t)=0.5916079783099616
Sampling t=0, np.sqrt(beta_t)=0.31622776601683794
Sampling t=2, np.sqrt(beta_t)=0.9219544457292886
Sampling t=1, np.sqrt(beta_t)=0.5916079783099616
Sampling t=0, np.sqrt(beta_t)=0.31622776601683794
Sampling t=2, np.sqrt(beta_t)=0.9219544457292886
Sampling t=1, np.sqrt(beta_t)=0.5916079783099616
Sampling t=0, np.sqrt(beta_t)=0.31622776601683794
Sampling t=2, np.sqrt(beta_t)=0.9219544457292886
Sampling t=1, np.sqrt(beta_t)=0.5916079783099616





Synthetic Generation: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3691/3691 [04:39<00:00, 13.21it/s]


Sampling t=0, np.sqrt(beta_t)=0.31622776601683794
Sampling t=2, np.sqrt(beta_t)=0.9219544457292886
Sampling t=1, np.sqrt(beta_t)=0.5916079783099616
Sampling t=0, np.sqrt(beta_t)=0.31622776601683794
Sampling t=2, np.sqrt(beta_t)=0.9219544457292886
Sampling t=1, np.sqrt(beta_t)=0.5916079783099616
Sampling t=0, np.sqrt(beta_t)=0.31622776601683794
Sampling t=2, np.sqrt(beta_t)=0.9219544457292886
Sampling t=1, np.sqrt(beta_t)=0.5916079783099616
Sampling t=0, np.sqrt(beta_t)=0.31622776601683794
Saving fake data...




Synths: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [59:36<00:00, 1788.41s/it]

Repeats: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [59:45<00:00, 3585.59s/it]


Time Elapsed: 59.76160706679026 min
